# 02 · Preparação dos dados

Do arquivo bruto ao conjunto de treino e teste:

1. Tradução das colunas e reconstrução da data/hora
2. Limpeza (tipos, ausentes, duplicados, faixas válidas)
3. Criação de variáveis (features)
4. Definição do alvo: lentidão da **próxima** janela de 30 minutos
5. Divisão treino/teste **temporal** (segunda a quinta para treino, sexta para teste)

## 1. Carga e tradução

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

# Arquivo original da CET/UCI: separador ';' e decimal ','
bruto = pd.read_csv('dados/bruto/sp_trafego_uci_2009.csv', sep=';', decimal=',')

RENOMEAR = {
    'Hour (Coded)': 'janela', 'Immobilized bus': 'onibus_parado', 'Broken Truck': 'caminhao_quebrado',
    'Vehicle excess': 'excesso_veiculos', 'Accident victim': 'acidente_com_vitima', 'Running over': 'atropelamento',
    'Fire vehicles': 'incendio_veiculo', 'Occurrence involving freight': 'ocorrencia_carga',
    'Incident involving dangerous freight': 'carga_perigosa', 'Lack of electricity': 'falta_energia',
    'Fire': 'incendio', 'Point of flooding': 'ponto_alagamento', 'Manifestations': 'manifestacao',
    'Defect in the network of trolleybuses': 'defeito_rede_trolebus', 'Tree on the road': 'arvore_na_via',
    'Semaphore off': 'semaforo_apagado', 'Intermittent Semaphore': 'semaforo_intermitente',
    'Slowness in traffic (%)': 'lentidao_pct',
}
OCORRENCIAS = [c for c in RENOMEAR.values() if c not in ('janela', 'lentidao_pct')]
DIAS = ['segunda', 'terca', 'quarta', 'quinta', 'sexta']

df = bruto.rename(columns=RENOMEAR)
# As linhas estão em ordem cronológica: 5 dias úteis x 27 janelas de 30 min (7h00 a 20h00),
# começando na segunda 14/12/2009.
dia_idx = df.index // 27
df['timestamp'] = (pd.Timestamp('2009-12-14 07:00')
                   + pd.to_timedelta(dia_idx, unit='D')
                   + pd.to_timedelta((df['janela'] - 1) * 30, unit='min'))
df['dia_semana'] = df['timestamp'].dt.dayofweek.map(dict(enumerate(DIAS)))
df['hora_decimal'] = df['timestamp'].dt.hour + df['timestamp'].dt.minute / 60
print(df.shape)

(135, 21)


## 2. Limpeza

In [2]:
antes = len(df)
df = df.drop_duplicates()
assert df['janela'].between(1, 27).all(), 'hora codificada fora da faixa'
assert df['lentidao_pct'].between(0, 100).all(), 'percentual inválido'
assert (df[OCORRENCIAS] >= 0).all().all(), 'contagem negativa'
df[OCORRENCIAS] = df[OCORRENCIAS].astype(int)
print(f'Linhas: {antes} -> {len(df)} | ausentes: {df.isna().sum().sum()}')

Linhas: 135 -> 135 | ausentes: 0


## 3. Criação de variáveis

| Variável | Por quê |
| --- | --- |
| `hora_decimal` | o horário é o fator mais forte (EDA) |
| `dia_semana_num` | a lentidão cresce ao longo da semana |
| `horario_pico` | marca 7h–10h e 17h–20h |
| `lentidao_atual` | a lentidão agora explica a próxima janela (corr. ≈ 0,95) |
| `variacao_30min` | se o trânsito está piorando ou melhorando |
| `total_ocorrencias`, `ocorrencias_semaforo` | resumem as 16 ocorrências, que são raras individualmente |

In [3]:
df['dia_semana_num'] = df['timestamp'].dt.dayofweek
df['data'] = df['timestamp'].dt.date
df['horario_pico'] = (df['hora_decimal'].between(7, 10) | df['hora_decimal'].between(17, 20)).astype(int)
df['total_ocorrencias'] = df[OCORRENCIAS].sum(axis=1)
df['ocorrencias_semaforo'] = df['semaforo_apagado'] + df['semaforo_intermitente']

df['lentidao_atual'] = df['lentidao_pct']
df['variacao_30min'] = df.groupby('data')['lentidao_pct'].diff()

## 4. Alvo: lentidão da próxima janela

O semáforo precisa se antecipar, então queremos a lentidão **30 minutos à frente**, sempre dentro do mesmo dia (o shift não atravessa a noite). Guardamos duas colunas:

- `lentidao_prox_30min`: o valor que queremos acertar (usado para avaliar);
- `variacao_prox_30min`: quanto ela muda em relação a agora (usado para treinar). A previsão final é `lentidao_atual + variação prevista`. Em cada dia, a primeira janela não tem `variacao_30min` e a última não tem alvo; essas linhas saem.

In [4]:
df['lentidao_prox_30min'] = df.groupby('data')['lentidao_pct'].shift(-1)
# Alvo de treino: quanto a lentidão vai SUBIR ou DESCER nos próximos 30 min.
# Prever a variação, e não o nível, força o modelo a aprender o padrão do dia
# em vez de só copiar a lentidão atual (ver notebook 03).
df['variacao_prox_30min'] = df['lentidao_prox_30min'] - df['lentidao_atual']
df_modelo = df.dropna(subset=['variacao_30min', 'lentidao_prox_30min']).reset_index(drop=True)
print(f'{len(df)} -> {len(df_modelo)} linhas (2 por dia removidas)')

135 -> 125 linhas (2 por dia removidas)


In [5]:
FEATURES = ['hora_decimal', 'dia_semana_num', 'horario_pico', 'lentidao_atual', 'variacao_30min',
            'total_ocorrencias', 'ocorrencias_semaforo'] + OCORRENCIAS
ALVO = 'lentidao_prox_30min'
ALVO_TREINO = 'variacao_prox_30min'
df_modelo[['timestamp', 'dia_semana'] + FEATURES[:7] + [ALVO, ALVO_TREINO]].head()

,timestamp,dia_semana,hora_decimal,dia_semana_num,horario_pico,lentidao_atual,variacao_30min,total_ocorrencias,ocorrencias_semaforo,lentidao_prox_30min,variacao_prox_30min
0,2009-12-14 07:30:00,segunda,7.5,0,1,6.6,2.5,0,0,8.7,2.1
1,2009-12-14 08:00:00,segunda,8.0,0,1,8.7,2.1,0,0,9.2,0.5
2,2009-12-14 08:30:00,segunda,8.5,0,1,9.2,0.5,0,0,11.1,1.9
3,2009-12-14 09:00:00,segunda,9.0,0,1,11.1,1.9,0,0,10.9,-0.2
4,2009-12-14 09:30:00,segunda,9.5,0,1,10.9,-0.2,0,0,8.3,-2.6


## 5. Divisão treino/teste temporal

Uma divisão aleatória colocaria janelas vizinhas do mesmo dia no treino e no teste, e a nota sairia otimista. Usamos o tempo: **treino = segunda a quinta**, **teste = sexta**, um dia inteiro que o modelo nunca viu. A validação cruzada do notebook 03 acontece só dentro do treino, separando por dia.

In [6]:
treino = df_modelo[df_modelo['dia_semana'] != 'sexta']
teste = df_modelo[df_modelo['dia_semana'] == 'sexta']
print(f'Treino: {len(treino)} linhas ({treino["dia_semana"].unique().tolist()})')
print(f'Teste:  {len(teste)} linhas ({teste["dia_semana"].unique().tolist()})')
print('Média do alvo  treino: %.2f  teste: %.2f' % (treino[ALVO].mean(), teste[ALVO].mean()))

Treino: 100 linhas (['segunda', 'terca', 'quarta', 'quinta'])
Teste:  25 linhas (['sexta'])
Média do alvo  treino: 10.16  teste: 11.78


Não há escalonamento: Random Forest trabalha com divisões por limiar e não depende da escala das variáveis.

In [7]:
COLS = ['timestamp', 'dia_semana'] + FEATURES + [ALVO, ALVO_TREINO]
df_modelo[COLS].to_csv('dados/sp_trafego.csv', index=False)
treino[COLS].to_csv('dados/treino.csv', index=False)
teste[COLS].to_csv('dados/teste.csv', index=False)
print('Salvos: dados/sp_trafego.csv, dados/treino.csv, dados/teste.csv')

Salvos: dados/sp_trafego.csv, dados/treino.csv, dados/teste.csv
